# 01 — Getting Connected

This notebook logs you in to OpenShift, grabs your API token, calls the AI models endpoint, and connects to S3-compatible storage (MinIO).

Run each cell from top to bottom by pressing **Shift + Enter**.

## Step 1 — Log in to OpenShift

Fill in your username below. The server address is detected automatically from the environment. Your password will be entered securely (it won't be saved in the notebook).

In [ ]:
import subprocess
import getpass
import os

host = os.environ["KUBERNETES_SERVICE_HOST"]
port = os.environ["KUBERNETES_SERVICE_PORT"]
OCP_SERVER = f"https://{host}:{port}"

OCP_USER = "user1"   # update to your username
OCP_PASS = getpass.getpass("Enter your OpenShift password: ")

result = subprocess.run(
    ["oc", "login", OCP_SERVER,
     "--username", OCP_USER,
     "--password", OCP_PASS,
     "--insecure-skip-tls-verify"],
    capture_output=True, text=True,
    cwd="/tmp"
)

print(result.stdout)
if result.returncode != 0:
    print("Login failed:", result.stderr)

## Step 2 — Confirm you are logged in

In [ ]:
!oc whoami
!oc project

## Step 3 — Get your API token

`oc whoami -t` prints the bearer token for your current login session. We will use it to call the AI models API.

In [ ]:
TOKEN = subprocess.run(
    ["oc", "whoami", "-t"], capture_output=True, text=True
).stdout.strip()

print("Token obtained:", TOKEN[:20], "...")

## Step 4 — List available AI models

In [ ]:
import requests
import json

MAAS_API_URL = "https://maas.apps.ocp.cloud.rhai-tmm.dev"

response = requests.get(
    f"{MAAS_API_URL}/maas-api/v1/models",
    headers={"Authorization": f"Bearer {TOKEN}"},
    verify=False
)

models = response.json()
print(json.dumps(models, indent=2))

## Step 5 — Connect to S3 storage (MinIO)

MinIO is an S3-compatible object store. The default credentials for this environment are **minio / minio1234**.

Update `MINIO_ENDPOINT` below to match your MinIO service URL.

In [ ]:
!pip install boto3 --user -q

In [ ]:
import boto3

MINIO_ENDPOINT   = "http://minio:9000"   # update this URL
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio1234"

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-2"
)

buckets = s3.list_buckets().get("Buckets", [])
print("Existing buckets:", [b["Name"] for b in buckets])

if "models" not in [b["Name"] for b in buckets]:
    s3.create_bucket(Bucket="models")
    print("Created bucket: models")
else:
    print("Bucket 'models' already exists")